# DINO-WM `Wall` Task Code Walkthrough

This notebook does one thing only: it reorganizes the key code for the `Wall` task into 8 cells so that the training pipeline and the entry points for later model modifications are immediately clear.

The reading order follows the actual code flow:

1. Setup
2. Compose the Wall configuration
3. Load the Wall dataset
4. Instantiate each submodule
5. Assemble the `VWorldModel`
6. Trace tensor shapes step by step
7. Inspect the Wall environment interface

## Cell 1: Setup

**Source files covered in this cell**
- `external/dino_wm/train.py`
- `external/dino_wm/models/...`
- `external/dino_wm/datasets/...`

**What this cell does**
- Locate the repository root and the `dino_wm` root directory.
- Add `dino_wm` to `sys.path`.
- Import the core libraries that will be used throughout the notebook.

**Why it is written this way**
- In the original repository, `train.py` is meant to be executed from the `dino_wm` root directory.
- This notebook lives under `notebooks/`, so without manually updating `sys.path`, later imports from `models`, `datasets`, and `env` will fail.

In [2]:
from pathlib import Path
import sys
import os

import torch
import hydra
from omegaconf import OmegaConf

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
dino_root = repo_root / "team9-model-code" / "external" / "dino_wm"
os.environ["DATASET_DIR"] = "/mnt/d/Embodied Vision group proj/team9-model-code/external/dino_wm/datasets"
print(os.environ["DATASET_DIR"])

if str(dino_root) not in sys.path:
    sys.path.insert(0, str(dino_root))

print("repo_root:", repo_root)
print("dino_root:", dino_root)
print("torch version:", torch.__version__)
print(os.environ.get("LD_LIBRARY_PATH"))

/mnt/d/Embodied Vision group proj/team9-model-code/external/dino_wm/datasets
repo_root: /mnt/d/Embodied Vision group proj
dino_root: /mnt/d/Embodied Vision group proj/team9-model-code/external/dino_wm
torch version: 2.3.0+cu121
:/home/mixian/.mujoco/mujoco210/bin:/usr/lib/nvidia:/home/mixian/.mujoco/mujoco210/bin:/home/mixian/.mujoco/mujoco210/bin:/home/mixian/.mujoco/mujoco210/bin:/usr/lib/nvidia


## Cell 2: Compose the Wall Configuration

**Source files covered in this cell**
- `external/dino_wm/conf/train.yaml`
- `external/dino_wm/conf/env/wall.yaml`
- `external/dino_wm/conf/encoder/dino.yaml`
- `external/dino_wm/conf/predictor/vit.yaml`
- `external/dino_wm/conf/decoder/vqvae.yaml`

**What this cell does**
- Re-compose the training configuration using Hydra.
- Change the default environment from `point_maze` to `wall`.
- Print only the most important fields instead of expanding the entire YAML.

**Why it is written this way**
- DINO-WM does not build the model by manually calling constructors first. It composes a config, then instantiates modules from that config.
- If you do not inspect the final composed config first, it is very easy to misjudge which modules are actually being used in the `Wall` task.

In [3]:
from hydra import initialize_config_dir, compose

conf_dir = str(dino_root / "conf")

with initialize_config_dir(config_dir=conf_dir, version_base=None):
    cfg = compose(
        config_name="train",
        overrides=[
            "env=wall",
            "frameskip=5",
            "num_hist=3",
            "num_pred=1",
        ],
    )

print("env:", cfg.env.name)
print("dataset target:", cfg.env.dataset._target_)
print("encoder:", cfg.encoder._target_)
print("encoder name:", cfg.encoder.name)
print("feature key:", cfg.encoder.feature_key)
print("predictor:", cfg.predictor._target_)
print("decoder:", cfg.decoder._target_)
print("img_size:", cfg.img_size)
print("num_hist:", cfg.num_hist)
print("num_pred:", cfg.num_pred)
print("frameskip:", cfg.frameskip)
print("concat_dim:", cfg.concat_dim)


env: wall
dataset target: datasets.wall_dset.load_wall_slice_train_val
encoder: models.dino.DinoV2Encoder
encoder name: dinov2_vits14
feature key: x_norm_patchtokens
predictor: models.vit.ViTPredictor
decoder: models.vqvae.VQVAE
img_size: 224
num_hist: 3
num_pred: 1
frameskip: 5
concat_dim: 1


## Cell 3: Load the Wall Dataset

**Source files covered in this cell**
- `external/dino_wm/datasets/wall_dset.py`
- `external/dino_wm/datasets/traj_dset.py`
- `external/dino_wm/datasets/img_transforms.py`

**What this cell does**
- Call the dataset factory specified by `cfg.env.dataset`.
- Inspect what a single training slice actually returns.
- Print the shapes of `visual`, `proprio`, `act`, and `state`.

**Why it is written this way**
- `wall_dset.py` loads full rollouts.
- What actually goes into training is the windowed slice produced by `traj_dset.py`.
- The most important detail here is that the action has already been concatenated along the `frameskip` dimension, so the action encoder input dimension is not the original `2`, but `2 * frameskip`.

In [4]:
datasets, traj_dsets = hydra.utils.call(
    cfg.env.dataset,
    num_hist=cfg.num_hist,
    num_pred=cfg.num_pred,
    frameskip=cfg.frameskip,
)

train_slice = datasets["train"]
val_slice = datasets["valid"]
train_traj = traj_dsets["train"]

print("num train slices:", len(train_slice))
print("num val slices:", len(val_slice))
print("num train trajectories:", len(train_traj))
print("slice action_dim:", train_slice.action_dim)
print("slice proprio_dim:", train_slice.proprio_dim)
print("slice state_dim:", train_slice.state_dim)

obs, act, state = train_slice[0]
print("obs keys:", list(obs.keys()))
print("visual shape:", obs["visual"].shape)
print("proprio shape:", obs["proprio"].shape)
print("act shape:", act.shape)
print("state shape:", state.shape)


/home/mixian/miniconda3/envs/dino_wm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading wall dataset...
Loaded 1920 rollouts
[[102, 33, 1056, 1591, 354, 245, 790, 1323, 142, 1745, 1588, 1473, 38, 1192, 735, 832, 547, 1186, 843, 660, 869, 83, 695, 1629, 179, 1658, 157, 1758, 1747, 250, 312, 1361, 1163, 1798, 275, 1099, 1787, 964, 1066, 609, 1666, 774, 181, 1502, 407, 668, 593, 1623, 1101, 565, 1350, 1572, 1094, 512, 148, 809, 263, 1853, 908, 927, 626, 569, 1730, 757, 1574, 1866, 1107, 1706, 391, 1255, 947, 828, 1720, 1057, 19, 1364, 1575, 1914, 1562, 1909, 534, 973, 538, 1017, 646, 285, 309, 1884, 1059, 552, 697, 1702, 145, 819, 649, 665, 169, 1051, 171, 1529, 1470, 291, 1371, 1240, 1641, 1746, 1443, 1307, 1801, 369, 1365, 525, 167, 1121, 1430, 874, 528, 1459, 1617, 770, 359, 923, 507, 884, 1616, 1314, 1149, 1379, 891, 1442, 1047, 1887, 248, 1767, 1620, 283, 392, 1290, 1650, 242, 1399, 807, 159, 204, 775, 1827, 500, 1352, 700, 1265, 1102, 149, 1440, 1258, 1738, 347, 690, 468, 1726, 1234, 1126, 868, 1501, 621, 585, 1523, 121, 1108, 846, 1167, 1405, 114, 474, 1406, 1

## Cell 4: Instantiate the Submodules

**Source files covered in this cell**
- `external/dino_wm/train.py` in `Trainer.init_models()`
- `external/dino_wm/models/dino.py`
- `external/dino_wm/models/proprio.py`
- `external/dino_wm/models/vit.py`
- `external/dino_wm/models/vqvae.py`

**What this cell does**
- Manually instantiate the `encoder`, `proprio_encoder`, `action_encoder`, `predictor`, and `decoder`.
- Make it explicit how the predictor’s `num_patches` and `dim` are determined.

**Why it is written this way**
- This step extracts `Trainer.init_models()` from the larger training script and isolates it.
- Looking at this layer separately makes it much easier to understand the visual encoder and the temporal predictor as two distinct components.

In [5]:
encoder = hydra.utils.instantiate(cfg.encoder)

proprio_encoder = hydra.utils.instantiate(
    cfg.proprio_encoder,
    in_chans=train_slice.proprio_dim,
    emb_dim=cfg.proprio_emb_dim,
)

action_encoder = hydra.utils.instantiate(
    cfg.action_encoder,
    in_chans=train_slice.action_dim,
    emb_dim=cfg.action_emb_dim,
)

if encoder.latent_ndim == 1:
    num_patches = 1
else:
    num_patches = (cfg.img_size // 16) ** 2

if cfg.concat_dim == 0:
    num_patches += 2

predictor_dim = encoder.emb_dim + (
    (cfg.proprio_emb_dim * cfg.num_proprio_repeat)
    + (cfg.action_emb_dim * cfg.num_action_repeat)
) * cfg.concat_dim

predictor = hydra.utils.instantiate(
    cfg.predictor,
    num_patches=num_patches,
    num_frames=cfg.num_hist,
    dim=predictor_dim,
)

decoder = hydra.utils.instantiate(
    cfg.decoder,
    emb_dim=encoder.emb_dim,
)

print(type(encoder), "emb_dim=", encoder.emb_dim, "latent_ndim=", encoder.latent_ndim)
print(type(proprio_encoder), "emb_dim=", proprio_encoder.emb_dim)
print(type(action_encoder), "emb_dim=", action_encoder.emb_dim)
print(type(predictor), "predictor_dim=", predictor_dim, "num_patches=", num_patches)
print(type(decoder))


Using cache found in /home/mixian/.cache/torch/hub/facebookresearch_dinov2_main
/home/mixian/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/mixian/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/mixian/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


using 3d prop position use_3d_pos=False
using 3d prop position use_3d_pos=False
<class 'models.dino.DinoV2Encoder'> emb_dim= 384 latent_ndim= 2
<class 'models.proprio.ProprioceptiveEmbedding'> emb_dim= 10
<class 'models.proprio.ProprioceptiveEmbedding'> emb_dim= 10
<class 'models.vit.ViTPredictor'> predictor_dim= 404 num_patches= 196
<class 'models.vqvae.VQVAE'>


## Cell 5: Assemble the `VWorldModel`

**Source file covered in this cell**
- `external/dino_wm/models/visual_world_model.py`

**What this cell does**
- Assemble the full world model from the submodules created in the previous cell.
- Run a single `forward()` pass with a minimal batch.
- Print `z_pred`, `visual_pred`, `visual_recon`, and `loss_components`.

**Why it is written this way**
- `visual_world_model.py` is where the actual training objectives come together.
- Running one full forward pass makes it much easier than source reading alone to see what the model is really optimizing.

In [6]:
from models.visual_world_model import VWorldModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = hydra.utils.instantiate(
    cfg.model,
    encoder=encoder,
    proprio_encoder=proprio_encoder,
    action_encoder=action_encoder,
    predictor=predictor,
    decoder=decoder,
    proprio_dim=cfg.proprio_emb_dim,
    action_dim=cfg.action_emb_dim,
    concat_dim=cfg.concat_dim,
    num_action_repeat=cfg.num_action_repeat,
    num_proprio_repeat=cfg.num_proprio_repeat,
).to(device)

model.eval()

batch_obs = {k: v.unsqueeze(0).to(device) for k, v in obs.items()}
batch_act = act.unsqueeze(0).to(device)

print("model device:", next(model.parameters()).device)
print("visual device:", batch_obs["visual"].device)
print("proprio device:", batch_obs["proprio"].device)
print("act device:", batch_act.device)

with torch.no_grad():
    z_pred, visual_pred, visual_recon, loss, loss_components = model(batch_obs, batch_act)

print("z_pred:", None if z_pred is None else z_pred.shape)
print("visual_pred:", None if visual_pred is None else visual_pred.shape)
print("visual_recon:", None if visual_recon is None else visual_recon.shape)
print("loss:", None if loss is None else float(loss))
print("loss keys:", [] if loss_components is None else list(loss_components.keys()))


device: cuda
num_action_repeat: 1
num_proprio_repeat: 1
proprio encoder: ProprioceptiveEmbedding(
  (patch_embed): Conv1d(2, 10, kernel_size=(1,), stride=(1,))
)
action encoder: ProprioceptiveEmbedding(
  (patch_embed): Conv1d(10, 10, kernel_size=(1,), stride=(1,))
)
proprio_dim: 10, after repeat: 10
action_dim: 10, after repeat: 10
emb_dim: 404
Model emb_dim:  404
model device: cuda:0
visual device: cuda:0
proprio device: cuda:0
act device: cuda:0
z_pred: torch.Size([1, 3, 196, 404])
visual_pred: torch.Size([1, 3, 3, 224, 224])
visual_recon: torch.Size([1, 4, 3, 224, 224])
loss: 3.8958396911621094
loss keys: ['decoder_recon_loss_pred', 'decoder_vq_loss_pred', 'decoder_loss_pred', 'z_loss', 'z_visual_loss', 'z_proprio_loss', 'decoder_recon_loss_reconstructed', 'decoder_vq_loss_reconstructed', 'decoder_loss_reconstructed', 'loss']


## Cell 6: Trace Tensor Shapes Step by Step

**Source file covered in this cell**
- `external/dino_wm/models/visual_world_model.py`

**What this cell does**
- Call `encode_obs()`, `encode_act()`, `encode()`, `predict()`, `separate_emb()`, and `decode_obs()` one by one.
- Print the tensor shapes at each step.

**Why it is written this way**
- This is the most important cell for later model modifications.
- If you want to replace DINO patch tokens with Slot tokens, this cell shows exactly which interfaces must remain compatible.

In [7]:
with torch.no_grad():
    z_obs = model.encode_obs(batch_obs)
    act_emb = model.encode_act(batch_act)
    z = model.encode(batch_obs, batch_act)
    z_src = z[:, :cfg.num_hist]
    z_roll = model.predict(z_src)
    z_obs_pred, z_act_pred = model.separate_emb(z_roll)
    decoded_obs, diff = model.decode_obs(z_obs_pred)

print("encoded visual:", z_obs["visual"].shape)
print("encoded proprio:", z_obs["proprio"].shape)
print("encoded action:", act_emb.shape)
print("joint z:", z.shape)
print("history z_src:", z_src.shape)
print("predicted z_roll:", z_roll.shape)
print("pred visual:", z_obs_pred["visual"].shape)
print("pred proprio:", z_obs_pred["proprio"].shape)
print("pred action:", z_act_pred.shape)
print("decoded visual:", decoded_obs["visual"].shape)
print("vq diff shape:", diff.shape)


encoded visual: torch.Size([1, 4, 196, 384])
encoded proprio: torch.Size([1, 4, 10])
encoded action: torch.Size([1, 4, 10])
joint z: torch.Size([1, 4, 196, 404])
history z_src: torch.Size([1, 3, 196, 404])
predicted z_roll: torch.Size([1, 3, 196, 404])
pred visual: torch.Size([1, 3, 196, 384])
pred proprio: torch.Size([1, 3, 10])
pred action: torch.Size([1, 3, 10])
decoded visual: torch.Size([1, 3, 3, 224, 224])
vq diff shape: torch.Size([1, 1])


## Cell 7: Inspect the Wall Environment Interface

**Source files covered in this cell**
- `external/dino_wm/env/__init__.py`
- `external/dino_wm/env/wall/wall_env_wrapper.py`
- `external/dino_wm/env/wall/envs/wall.py`
- `external/dino_wm/plan.py`

**What this cell does**
- Register the environment and call `gym.make("wall")`.
- Check the inputs and outputs of `prepare()` and `rollout()`.

**Why it is written this way**
- During training, the code reads from an offline dataset.
- During planning, it returns to the actual environment through the environment wrapper.
- Looking at both sides together makes the full closed loop of the `Wall` task much clearer.

In [8]:
import os
import warnings

os.environ["D4RL_SUPPRESS_IMPORT_ERROR"] = "1"

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="gym.envs.registration")
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")
warnings.filterwarnings("ignore", message=".*declare_namespace.*")

import gym
import env

wall_env = gym.make("wall")
obs0, state0 = wall_env.prepare(seed=0, init_state=torch.tensor([10.0, 20.0]))

print("prepare visual:", obs0["visual"].shape)
print("prepare state:", state0.shape)

actions = torch.tensor([
    [1.0, 0.0],
    [1.0, 0.5],
    [0.5, 0.5],
], dtype=torch.float32)

rollout_obs, rollout_states = wall_env.rollout(
    seed=0,
    init_state=state0.numpy(),
    actions=actions.numpy(),
)

print("rollout visual:", rollout_obs["visual"].shape)
print("rollout proprio:", rollout_obs["proprio"].shape)
print("rollout states:", rollout_states.shape)


/home/mixian/miniconda3/envs/dino_wm/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:15: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
No module named 'mjrl'
No module named 'flow'
No module named 'carla'
pybullet build time: Jan 29 2025 23:16:28
/home/mixian/miniconda3/envs/dino_wm/lib/python3.10/site-packages/pybullet_envs/env_bases.py:8: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
/home/mixian/miniconda3/envs/dino_wm/lib/python3.10/site-packages/pkg_resources/__init__.py:2825: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as 

prepare visual: torch.Size([224, 224, 3])
prepare state: torch.Size([2])
rollout visual: (4, 224, 224, 3)
rollout proprio: (4, 2)
rollout states: (4, 2)


/mnt/d/Embodied Vision group proj/team9-model-code/external/dino_wm/env/wall/envs/wall.py:296: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.reset_to_state = torch.tensor(init_state)


## Cell 8: Where Should Slot Attention Be Inserted?

**Current visual pipeline**
`image -> DinoV2Encoder -> patch tokens -> VWorldModel.encode_obs() -> predictor`

**Not decided yet**